# MML Chapter 5 — Vector Calculus

> Differentiation is asking "what is the best linear approximation here?"  
> Everything — partial derivatives, Jacobians, backprop — is that one idea applied at different scales.

**Sections:**
1. Numerical vs Analytical Gradient — $f(x) = x_1^2 + 2x_2^2$
2. Jacobian of a Vector Function — compute numerically and analytically
3. Hessian — compute numerically; verify symmetry; check positive definiteness
4. Gradient Descent on a 2D Function — implement from scratch; plot trajectory
5. Backpropagation — tiny 2-layer network; verify gradients with finite differences
6. Taylor Approximation — $\sin(x)$ vs 1st, 2nd, 4th order Taylor around $x=0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.family': 'sans-serif',
})
print('Imports done.')

---
## 1. Numerical Gradient vs Analytical Gradient

Consider $f(\mathbf{x}) = x_1^2 + 2x_2^2$.

**Analytical gradient:**
$$\nabla f = \begin{bmatrix}\partial f/\partial x_1 \\ \partial f/\partial x_2\end{bmatrix} = \begin{bmatrix}2x_1 \\ 4x_2\end{bmatrix}$$

**Numerical gradient** (central finite differences, more accurate than forward differences):
$$\frac{\partial f}{\partial x_i} \approx \frac{f(\mathbf{x} + h\mathbf{e}_i) - f(\mathbf{x} - h\mathbf{e}_i)}{2h}$$

In [ ]:
def f(x):
    """f(x) = x1^2 + 2*x2^2"""
    return x[0]**2 + 2 * x[1]**2

def analytical_grad(x):
    """Analytical gradient of f."""
    return np.array([2 * x[0], 4 * x[1]])

def numerical_grad(func, x, h=1e-5):
    """Central finite differences gradient."""
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        e_i = np.zeros_like(x, dtype=float)
        e_i[i] = 1.0
        grad[i] = (func(x + h * e_i) - func(x - h * e_i)) / (2 * h)
    return grad

# Evaluate at a few test points
test_points = [
    np.array([1.0, 2.0]),
    np.array([-3.0, 0.5]),
    np.array([0.0, 0.0]),
    np.array([2.5, -1.5]),
]

print(f'{"Point x":22s}  {"Analytical ∇f":25s}  {"Numerical ∇f":25s}  {"Max abs diff":>12s}')
print('-' * 95)
for x in test_points:
    ag = analytical_grad(x)
    ng = numerical_grad(f, x)
    diff = np.max(np.abs(ag - ng))
    print(f'{str(x):22s}  {str(np.round(ag,6)):25s}  {str(np.round(ng,6)):25s}  {diff:.2e}')

In [ ]:
# --- Effect of step size h on numerical gradient accuracy ---
x0 = np.array([1.0, 2.0])
ag = analytical_grad(x0)

h_values = [10**(-k) for k in range(1, 14)]
errors = []
for h in h_values:
    ng = numerical_grad(f, x0, h=h)
    errors.append(np.max(np.abs(ag - ng)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(h_values, errors, 'o-', color='steelblue', lw=2, markersize=6)
ax.set_xlabel('Step size h')
ax.set_ylabel('Max |analytical - numerical| gradient')
ax.set_title('Numerical gradient accuracy vs step size h\n(central differences on f(x) = x₁² + 2x₂²)')
ax.grid(True, which='both', alpha=0.3)
# Annotate optimal region
best_h = h_values[np.argmin(errors)]
best_err = min(errors)
ax.annotate(f'Best h ≈ {best_h:.0e}\nerr = {best_err:.2e}',
            xy=(best_h, best_err), xytext=(best_h * 10, best_err * 100),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=9)
plt.tight_layout()
plt.show()
print('Note: very small h suffers from floating-point cancellation errors.')

---
## 2. Jacobian of a Vector Function

For $f: \mathbb{R}^n \to \mathbb{R}^m$, the Jacobian is:
$$J = \frac{\partial f}{\partial \mathbf{x}} \in \mathbb{R}^{m \times n}, \quad J_{ij} = \frac{\partial f_i}{\partial x_j}$$

**Example:** $f: \mathbb{R}^2 \to \mathbb{R}^2$,
$$f(\mathbf{x}) = \begin{bmatrix}x_1^2 \\ x_1 x_2\end{bmatrix}, \quad J = \begin{bmatrix}2x_1 & 0 \\ x_2 & x_1\end{bmatrix}$$

In [ ]:
def f_vec(x):
    """f: R^2 -> R^2, f(x) = [x1^2, x1*x2]"""
    return np.array([x[0]**2, x[0] * x[1]])

def analytical_jacobian(x):
    """Analytical Jacobian of f_vec."""
    return np.array([[2 * x[0], 0.0],
                     [x[1],     x[0]]])

def numerical_jacobian(func, x, h=1e-5):
    """Numerical Jacobian via central differences."""
    n  = len(x)
    f0 = func(x)
    m  = len(f0)
    J  = np.zeros((m, n))
    for j in range(n):
        e_j = np.zeros(n)
        e_j[j] = 1.0
        J[:, j] = (func(x + h * e_j) - func(x - h * e_j)) / (2 * h)
    return J

# Evaluate at a test point
x_test = np.array([2.0, 3.0])

J_anal = analytical_jacobian(x_test)
J_num  = numerical_jacobian(f_vec, x_test)

print(f'Test point x = {x_test}')
print(f'f(x) = {f_vec(x_test)}')
print('\nAnalytical Jacobian J:')
print(J_anal)
print('\nNumerical Jacobian J (central differences):')
print(np.round(J_num, 8))
print(f'\nMax abs difference: {np.max(np.abs(J_anal - J_num)):.2e}')
print(f'Match: {np.allclose(J_anal, J_num, atol=1e-8)}')

In [ ]:
# --- Visualize how the Jacobian linearizes f locally ---
x0  = np.array([1.5, 1.0])
J0  = analytical_jacobian(x0)
f0  = f_vec(x0)

# Sample a grid of delta_x and compare f(x0 + dx) vs J0 @ dx (linear approx)
deltas_1d = np.linspace(-0.5, 0.5, 20)
errors_lin = []
delta_norms = []

for d in deltas_1d:
    dx = np.array([d, 0.5 * d])
    f_true  = f_vec(x0 + dx)
    f_linear = f0 + J0 @ dx  # first-order Taylor
    errors_lin.append(np.linalg.norm(f_true - f_linear))
    delta_norms.append(np.linalg.norm(dx))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(delta_norms[1:], errors_lin[1:], 'o-', color='steelblue', lw=2, markersize=5)
# Reference line: O(||dx||^2) — linearization should be second-order accurate
dn = np.array(delta_norms[1:])
ax.loglog(dn, dn**2 * 0.5, 'r--', lw=1, label='O(‖Δx‖²) reference')
ax.set_xlabel('‖Δx‖')
ax.set_ylabel('‖f(x₀+Δx) - f(x₀) - J·Δx‖')
ax.set_title('Linearization error of Jacobian\n(should be O(‖Δx‖²) — first-order Taylor)')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Hessian

For $f: \mathbb{R}^n \to \mathbb{R}$, the Hessian captures curvature:
$$H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$$

The Hessian is always **symmetric** (Schwarz's theorem).  
At a critical point: $H \succ 0$ (positive definite) → local minimum.

In [ ]:
def g(x):
    """g(x) = x1^4 + x2^4 - 2*x1^2 - 4*x2^2 + x1*x2 + 6"""
    return x[0]**4 + x[1]**4 - 2*x[0]**2 - 4*x[1]**2 + x[0]*x[1] + 6

def numerical_hessian(func, x, h=1e-4):
    """Hessian via central finite differences."""
    n = len(x)
    H = np.zeros((n, n))
    f0 = func(x)
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n); ei[i] = 1.0
            ej = np.zeros(n); ej[j] = 1.0
            if i == j:
                # Second derivative: [f(x+h*ei) - 2f(x) + f(x-h*ei)] / h^2
                H[i, j] = (func(x + h*ei) - 2*f0 + func(x - h*ei)) / h**2
            else:
                # Mixed partial: [f(x+h*ei+h*ej) - f(x+h*ei-h*ej)
                #                  - f(x-h*ei+h*ej) + f(x-h*ei-h*ej)] / (4h^2)
                H[i, j] = (func(x + h*ei + h*ej)
                           - func(x + h*ei - h*ej)
                           - func(x - h*ei + h*ej)
                           + func(x - h*ei - h*ej)) / (4 * h**2)
    return H

# Evaluate Hessian at a test point
x_test = np.array([1.0, 1.5])
H = numerical_hessian(g, x_test)

print(f'Test point x = {x_test}')
print(f'g(x) = {g(x_test):.6f}')
print('\nNumerical Hessian H:')
print(np.round(H, 6))

# Symmetry check
print(f'\nH is symmetric: {np.allclose(H, H.T, atol=1e-8)}')
print(f'Max |H - Hᵀ|: {np.max(np.abs(H - H.T)):.2e}')

In [ ]:
# --- Check positive definiteness at a critical point ---
# For a simple convex function: h(x) = x1^2 + 3*x2^2  (minimum at origin)
def h_convex(x):
    return x[0]**2 + 3 * x[1]**2

x_min = np.array([0.0, 0.0])  # known minimum
H_min = numerical_hessian(h_convex, x_min)

eigenvalues_H = np.linalg.eigvalsh(H_min)  # symmetric → eigvalsh

print('h(x) = x₁² + 3x₂²  at x = (0, 0)')
print(f'Numerical Hessian at minimum:\n{np.round(H_min, 8)}')
print(f'\nEigenvalues of H: {eigenvalues_H}')
print(f'All eigenvalues > 0: {np.all(eigenvalues_H > 0)} → positive definite → local minimum')

# Also check at a saddle point of g
print('\n--- Hessian of g at x = (0, 0) (saddle-ish region) ---')
x_saddle = np.array([0.0, 0.0])
H_saddle = numerical_hessian(g, x_saddle)
eig_s = np.linalg.eigvalsh(H_saddle)
print(f'Hessian:\n{np.round(H_saddle, 6)}')
print(f'Eigenvalues: {np.round(eig_s, 6)}')
print(f'Mixed signs: {np.any(eig_s < 0) and np.any(eig_s > 0)} → saddle point')

---
## 4. Gradient Descent on a 2D Function

**Algorithm:**
$$\mathbf{x}_{t+1} = \mathbf{x}_t - \alpha \nabla f(\mathbf{x}_t)$$

where $\alpha > 0$ is the learning rate (step size). We minimize $f(x_1, x_2) = x_1^2 + 5x_2^2$ — an elliptic bowl with the minimum at the origin.

In [ ]:
def bowl(x):
    """f(x) = x1^2 + 5*x2^2  (min at origin)"""
    return x[0]**2 + 5 * x[1]**2

def bowl_grad(x):
    return np.array([2 * x[0], 10 * x[1]])

def gradient_descent(grad_fn, x_init, lr, n_steps):
    """Run gradient descent; return trajectory."""
    x = np.array(x_init, dtype=float)
    trajectory = [x.copy()]
    for _ in range(n_steps):
        x = x - lr * grad_fn(x)
        trajectory.append(x.copy())
    return np.array(trajectory)

# Compare three learning rates
x0     = np.array([3.0, 2.0])
n_steps = 40
configs = [
    (0.05,  'steelblue', 'lr = 0.05 (slow)'),
    (0.18,  'green',     'lr = 0.18 (good)'),
    (0.21,  'crimson',   'lr = 0.21 (oscillates)'),
]

trajs = [(gradient_descent(bowl_grad, x0, lr, n_steps), color, label)
          for lr, color, label in configs]

# Contour plot
X1, X2 = np.meshgrid(np.linspace(-3.5, 3.5, 200), np.linspace(-2.5, 2.5, 200))
Z = X1**2 + 5 * X2**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trajectory plot
ax = axes[0]
cs = ax.contour(X1, X2, Z, levels=np.logspace(-1, 1.3, 20), cmap='Blues', alpha=0.5)
ax.clabel(cs, inline=True, fontsize=7)

for traj, color, label in trajs:
    ax.plot(traj[:, 0], traj[:, 1], 'o-', color=color, markersize=4, lw=1.5, label=label)
    ax.plot(*traj[0], 's', color=color, markersize=8)

ax.plot(0, 0, '*', color='gold', markersize=14, zorder=10, label='Minimum (0, 0)')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Gradient descent trajectories\n$f(x) = x_1^2 + 5x_2^2$')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# Right: loss vs iteration
ax = axes[1]
for traj, color, label in trajs:
    losses = [bowl(x) for x in traj]
    ax.semilogy(losses, color=color, lw=2, label=label)
ax.set_xlabel('Iteration')
ax.set_ylabel('f(x) (log scale)')
ax.set_title('Loss vs iteration')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Gradient Descent: effect of learning rate', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Backpropagation — Tiny 2-Layer Network

We implement a minimal 2-layer network from scratch and verify the gradients using finite differences.

**Architecture:**  
$\mathbf{h} = \sigma(W_1 \mathbf{x} + \mathbf{b}_1)$, $\hat{y} = W_2 \mathbf{h} + b_2$, $L = \frac{1}{2}(\hat{y} - y)^2$

where $\sigma(z) = \tanh(z)$.

**Backprop** is the chain rule applied layer by layer, right to left.

In [ ]:
# ---- Network architecture ----
# Input:  x ∈ R^3
# Hidden: h ∈ R^4  (tanh activation)
# Output: y_hat ∈ R^1  (linear)

n_in, n_hid, n_out = 3, 4, 1

# Initialize parameters (small random)
W1 = rng.standard_normal((n_hid, n_in)) * 0.5
b1 = rng.standard_normal(n_hid) * 0.1
W2 = rng.standard_normal((n_out, n_hid)) * 0.5
b2 = rng.standard_normal(n_out) * 0.1

# Pack all parameters into a single flat vector for finite-difference check
def pack_params(W1, b1, W2, b2):
    return np.concatenate([W1.ravel(), b1.ravel(), W2.ravel(), b2.ravel()])

def unpack_params(theta):
    s1 = n_hid * n_in
    s2 = s1 + n_hid
    s3 = s2 + n_out * n_hid
    s4 = s3 + n_out
    W1 = theta[:s1].reshape(n_hid, n_in)
    b1 = theta[s1:s2]
    W2 = theta[s2:s3].reshape(n_out, n_hid)
    b2 = theta[s3:s4]
    return W1, b1, W2, b2

theta0 = pack_params(W1, b1, W2, b2)
print(f'Total parameters: {len(theta0)}')
print(f'  W1: {W1.shape}, b1: {b1.shape}, W2: {W2.shape}, b2: {b2.shape}')

In [ ]:
# ---- Forward and backward pass ----

def forward_backward(theta, x, y_true):
    """Compute loss and gradients via backprop. Returns (loss, grad_theta)."""
    W1, b1, W2, b2 = unpack_params(theta)

    # --- Forward pass ---
    z1    = W1 @ x + b1          # pre-activation, shape (n_hid,)
    h     = np.tanh(z1)           # hidden, shape (n_hid,)
    z2    = W2 @ h + b2          # output pre-activation, shape (n_out,)
    y_hat = z2                    # linear output, shape (n_out,)
    loss  = 0.5 * np.sum((y_hat - y_true) ** 2)  # scalar

    # --- Backward pass ---
    # dL/dz2
    dL_dz2 = (y_hat - y_true)    # shape (n_out,)

    # dL/dW2 = dL/dz2 ⊗ h  (outer product)
    dL_dW2 = np.outer(dL_dz2, h)  # shape (n_out, n_hid)
    # dL/db2
    dL_db2 = dL_dz2               # shape (n_out,)

    # Propagate through W2 to h
    dL_dh  = W2.T @ dL_dz2       # shape (n_hid,)

    # Through tanh: d/dz1 tanh(z1) = 1 - tanh(z1)^2 = sech^2(z1)
    dL_dz1 = dL_dh * (1 - h**2)  # shape (n_hid,), elementwise

    # dL/dW1 = dL/dz1 ⊗ x
    dL_dW1 = np.outer(dL_dz1, x)  # shape (n_hid, n_in)
    # dL/db1
    dL_db1 = dL_dz1               # shape (n_hid,)

    grad_theta = pack_params(dL_dW1, dL_db1, dL_dW2, dL_db2)
    return loss, grad_theta

# Test data
x_data  = np.array([0.5, -1.2, 0.8])
y_data  = np.array([0.3])

loss, grad_bp = forward_backward(theta0, x_data, y_data)
print(f'Loss: {loss:.6f}')
print(f'Backprop gradient (first 8 entries): {np.round(grad_bp[:8], 6)}')

In [ ]:
# ---- Verify gradients with finite differences ----

def loss_only(theta):
    W1, b1, W2, b2 = unpack_params(theta)
    z1    = W1 @ x_data + b1
    h     = np.tanh(z1)
    z2    = W2 @ h + b2
    return 0.5 * np.sum((z2 - y_data) ** 2)

h_fd = 1e-5
grad_fd = np.zeros_like(theta0)
for i in range(len(theta0)):
    e_i = np.zeros_like(theta0)
    e_i[i] = 1.0
    grad_fd[i] = (loss_only(theta0 + h_fd * e_i) - loss_only(theta0 - h_fd * e_i)) / (2 * h_fd)

max_diff   = np.max(np.abs(grad_bp - grad_fd))
rel_error  = np.linalg.norm(grad_bp - grad_fd) / (np.linalg.norm(grad_bp) + np.linalg.norm(grad_fd))

print('Gradient check: backprop vs finite differences')
print(f'  Max absolute difference: {max_diff:.2e}')
print(f'  Relative error:          {rel_error:.2e}')
print(f'  Gradients match (tol 1e-6): {np.allclose(grad_bp, grad_fd, atol=1e-6)}')

# Plot: backprop grad vs fd grad for each parameter
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(grad_bp, 'o', color='steelblue', markersize=5, alpha=0.8, label='Backprop gradient')
ax.plot(grad_fd, 'x', color='crimson',   markersize=6, alpha=0.7, label='Finite difference gradient')
ax.set_xlabel('Parameter index')
ax.set_ylabel('Gradient value')
ax.set_title('Backprop vs Finite Difference gradient verification')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Short training loop to confirm the network learns ----
# Generate a small dataset
n_data = 30
X_train = rng.standard_normal((n_data, n_in))
y_train = (X_train[:, 0] - 0.5 * X_train[:, 1] + 0.2 * X_train[:, 2]
           + 0.05 * rng.standard_normal(n_data))  # noisy linear signal

theta  = pack_params(W1.copy(), b1.copy(), W2.copy(), b2.copy())
lr_nn  = 0.01
losses = []

for epoch in range(300):
    total_loss = 0.0
    total_grad = np.zeros_like(theta)
    for xi, yi in zip(X_train, y_train):
        l, g = forward_backward(theta, xi, np.array([yi]))
        total_loss += l
        total_grad += g
    total_loss /= n_data
    total_grad /= n_data
    theta -= lr_nn * total_grad
    losses.append(total_loss)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='steelblue', lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('2-layer network training loss (gradient descent from scratch)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {losses[-1]:.6f}')

---
## 6. Taylor Approximation

Taylor expansion of $f$ around $x_0 = 0$ (Maclaurin series):
$$f(x) \approx \sum_{k=0}^{K} \frac{f^{(k)}(x_0)}{k!}(x - x_0)^k$$

For $f(x) = \sin(x)$:
- Order 1: $x$
- Order 2: $x - \frac{x^3}{6}$ (odd powers only, but we include up to x² explicitly)
- Order 4: $x - \frac{x^3}{6} + \frac{x^5}{120}$

Higher-order approximations are accurate over wider intervals.

In [ ]:
x = np.linspace(-4, 4, 500)

# True function
f_true = np.sin(x)

# Taylor approximations of sin(x) around x=0
# sin(x) = x - x^3/6 + x^5/120 - x^7/5040 + ...
taylor_1 = x                                                      # T1
taylor_3 = x - x**3 / 6                                          # T3 ("2nd order" in terms of derivatives used)
taylor_5 = x - x**3 / 6 + x**5 / 120                            # T5
taylor_7 = x - x**3 / 6 + x**5 / 120 - x**7 / 5040             # T7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: approximations
ax = axes[0]
ax.plot(x, f_true,  'k-',  lw=2.5, label='sin(x) (true)', zorder=5)
ax.plot(x, taylor_1, '--', lw=1.5, color='#e74c3c',  label='Order 1: x')
ax.plot(x, taylor_3, '--', lw=1.5, color='#e67e22',  label='Order 3: x - x³/6')
ax.plot(x, taylor_5, '--', lw=1.5, color='#27ae60',  label='Order 5: + x⁵/120')
ax.plot(x, taylor_7, '--', lw=1.5, color='#2980b9',  label='Order 7: - x⁷/5040')
ax.set_ylim(-2.5, 2.5)
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Taylor approximations of sin(x) around x = 0')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: approximation error |f(x) - T_k(x)|
ax = axes[1]
for tay, color, label in [
        (taylor_1, '#e74c3c', 'Order 1'),
        (taylor_3, '#e67e22', 'Order 3'),
        (taylor_5, '#27ae60', 'Order 5'),
        (taylor_7, '#2980b9', 'Order 7'),
    ]:
    ax.semilogy(x, np.abs(f_true - tay) + 1e-16, lw=1.5, color=color, label=label)
ax.set_xlabel('x')
ax.set_ylabel('|sin(x) - T_k(x)|')
ax.set_title('Approximation error (log scale)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Taylor Series: higher order → accurate over wider range', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Multivariate Taylor: 2nd-order approximation of a 2D function ---
# f(x1, x2) = sin(x1) * exp(-x2^2/2)
# Expand around x0 = (pi/4, 0)

def f2d(x):
    return np.sin(x[0]) * np.exp(-x[1]**2 / 2)

x0_2d = np.array([np.pi / 4, 0.0])

# Compute gradient and Hessian numerically at x0
grad0 = numerical_grad(f2d, x0_2d)
H0    = numerical_hessian(f2d, x0_2d)
f0_2d = f2d(x0_2d)

# Second-order Taylor approximation on a grid
xx1 = np.linspace(-0.5, 2.5, 150)
xx2 = np.linspace(-2.0, 2.0, 150)
XX1, XX2 = np.meshgrid(xx1, xx2)

F_true   = np.sin(XX1) * np.exp(-XX2**2 / 2)
F_taylor = np.zeros_like(F_true)

for i, x1 in enumerate(xx1):
    for j, x2 in enumerate(xx2):
        dx = np.array([x1, x2]) - x0_2d
        F_taylor[j, i] = f0_2d + grad0 @ dx + 0.5 * dx @ H0 @ dx

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

levels = np.linspace(-1, 1, 21)
kw = dict(levels=levels, cmap='RdBu_r', vmin=-1, vmax=1)

axes[0].contourf(XX1, XX2, F_true, **kw)
axes[0].plot(*x0_2d, 'k*', markersize=12, label='x₀')
axes[0].set_title('True: sin(x₁)·exp(-x₂²/2)')
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')

axes[1].contourf(XX1, XX2, F_taylor, **kw)
axes[1].plot(*x0_2d, 'k*', markersize=12, label='x₀')
axes[1].set_title('2nd-order Taylor around x₀ = (π/4, 0)')
axes[1].set_xlabel('x₁')

err_map = np.abs(F_true - F_taylor)
im = axes[2].contourf(XX1, XX2, err_map, levels=20, cmap='hot_r')
axes[2].plot(*x0_2d, 'b*', markersize=12)
axes[2].set_title('Approximation error |f - T₂|')
axes[2].set_xlabel('x₁')
plt.colorbar(im, ax=axes[2])

plt.suptitle('Multivariate Taylor Approximation (2nd order)', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Error is smallest near x₀ = ({x0_2d[0]:.3f}, {x0_2d[1]:.3f}) and grows with distance.')

---
## Summary

| Concept | Formula | Key Insight |
|---------|---------|-------------|
| Gradient $f: \mathbb{R}^n \to \mathbb{R}$ | $\nabla f \in \mathbb{R}^{1 \times n}$ | Direction of steepest ascent |
| Jacobian $f: \mathbb{R}^n \to \mathbb{R}^m$ | $J \in \mathbb{R}^{m \times n}$ | Best linear approximation to $f$ |
| Hessian | $H_{ij} = \partial^2 f / \partial x_i \partial x_j$ | Curvature; symmetric; PD at min |
| Gradient descent | $x \leftarrow x - \alpha \nabla f$ | First-order optimization |
| Backprop | Chain rule, reversed | Efficient gradient for neural nets |
| Taylor series | $f(x_0) + \nabla f \cdot \Delta x + \frac{1}{2}\Delta x^\top H \Delta x + \cdots$ | Local polynomial approximation |

**Core theme:** A derivative is a **linear map** (Jacobian matrix) that locally approximates a nonlinear function. Backpropagation is just the chain rule applied to a sequence of such linear maps, traversed in reverse.